# Area aggregation (point-to-polygon masking)

Replaces each original data point with the identifier of the 100 m population grid cell it falls into, and reports only aggregated case counts per cell. This removes exact coordinates entirely in favour of areal counts.

**Input:** the original point dataset, the population grid (`data/original`, `data/population_grid`) — see `README.md`.
**Output:** `data/area_aggregation/aggregated_points.shp`, one row per grid cell with publication flags for case counts below the k-anonymity threshold.


In [ ]:
# Imports
import os
from pathlib import Path
import geopandas as gpd
import numpy as np


In [ ]:
# Paths
# Data directory. Override with the DATA_DIR environment variable to
# point at a different copy of the data without editing this file, e.g.
#   DATA_DIR=/path/to/data jupyter execute this_notebook.ipynb
# The data itself is not part of this repository -- see README.md.
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))

ODP_PATH = DATA_DIR / "original" / "2008_eov_illness_stand.shp"
GRID_PATH = DATA_DIR / "population_grid" / "GHS_Hungary_pop_city-clipped2.shp"

OUT_DIR = DATA_DIR / "area_aggregation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "aggregated_points.shp"

#Column names
VISITS_COL = "VISITS"      # number of hospital visits
POP_COL = "VALUE"          # population count in each grid cell
POP_THRESHOLD = 5         # minimum population required for publication
CASE_THRESHOLD = 5        # minimum number of cases required for publication
ID_COL = "GID"         # unique grid cell ID

#Loading ODP and population grid
odp = gpd.read_file(ODP_PATH)
grid = gpd.read_file(GRID_PATH)


In [ ]:
#check if both layers have a CRS defined and are the same, if not reproject ODP to grid CRS
if grid.crs is None:
    raise ValueError("The grid layer has no CRS defined.")

if odp.crs != grid.crs:
    odp = odp.to_crs(grid.crs)

#check if ID_COL exists in grid
if ID_COL not in grid.columns:
    raise ValueError(
        f"{ID_COL} not found in grid layer."
    )

#Spatial join to assign grid cell IDs and population counts to ODP points
joined = gpd.sjoin(
    odp,
    grid[[ID_COL, "geometry", POP_COL]],
    how="left",
    predicate="intersects"
)

In [ ]:
#aggregate visit counts and population by grid cell
agg = (
    joined
    .groupby(ID_COL, dropna=False)
    .agg(
        n_cases=(VISITS_COL, "count"),
        visits_total=(VISITS_COL, "sum")
    )
    .reset_index()
)

cells = grid.merge(agg, on=ID_COL, how="left")

cells["n_cases"] = cells["n_cases"].fillna(0).astype(int)
cells["visits_total"] = cells["visits_total"].fillna(0).astype(int)
cells["pop_in_cell"] = cells[POP_COL].astype(float)


#calculate visit rate per capita, handle division by zero
cells["visits_rate"] = np.where(
    cells["pop_in_cell"] > 0,
    cells["visits_total"] / cells["pop_in_cell"],
    np.nan
)

#apply privacy rules to determine publishable cells
cells["is_publishable"] = (
    (cells["pop_in_cell"] >= POP_THRESHOLD) 
    & (cells["n_cases"] >= CASE_THRESHOLD)
)

cells["n_cases_public"] = np.where(
    cells["is_publishable"],
    cells["n_cases"],
    None
)

cells["visits_total_public"] = np.where(
    cells["is_publishable"],
    cells["visits_total"],
    None
)

cells["visits_rate_public"] = np.where(
    cells["is_publishable"],
    cells["visits_rate"],
    None
)

In [ ]:
#select only relevant columns for publication
public_layer = cells[
    [
        ID_COL,
        "geometry",
        "pop_in_cell",
        "is_publishable",
        "n_cases_public",
        "visits_total_public",
        "visits_rate_public",
    ]
].copy()

#save the resulting layer to a new shapefile
public_layer.to_file(OUT_PATH)
print(f"Done. Aggregated and anonymized data saved to:\n{OUT_PATH}")